SPT Map Manipulation
--------------------

We typically store maps for SPT in `.g3` files that contain `G3Frame` objects.  See the spt3g_software [documentation](https://southpoletelescope.github.io/spt3g_software/frames.html) and the other tutorial in this directory for details.  Here, I'll assume that you already know how to read frames from disk.  For details on map objects and the various pipeline modules described here, see [this page](https://southpoletelescope.github.io/spt3g_software/moddoc_maps.html).

In [ ]:
import numpy as np
from spt3g import core, cluster, maps

I'm going to be doing some operations on data that's stored on the grid, so I'll use the grid-friendly tools discussed in the `grid_tools` notebook.

Let's load a random winter field observation from disk.  I know that this file contains several other frames in it, so I'm going to use this shortcut to keep just the last frame, which I know contains the map we want:

In [ ]:
frame = list(cluster.GridFile("/sptgrid/data/onlinemaps/ra0hdec-44.75/294349321_150GHz_tonly.g3.gz"))[-1]

We're going to run into some interesting memory handling behavior, so I'm going to import the garbage collection package, `gc` and use it to collect any dangling objects that might be hanging on to memory:

In [ ]:
import gc
gc.collect()

The above way of reading a file can sometimes be problematic, if you have multiple high-memory frames in your file, because they will all be loaded in to memory with the above construction.  If you want just a specific frame, it is better practice to load it in a loop, like below.  Only one frame will stay in memory at a time, until you get to the one you want to keep.

In [ ]:
frame = None
for fr in cluster.GridFile("/sptgrid/data/onlinemaps/ra0hdec-44.75/294349321_150GHz_tonly.g3.gz"):
    if fr.type == core.G3FrameType.Map:
        frame = fr
        break

In [ ]:
print(frame)

This is a typical map frame containing some temperature-only data.  Data are stored with weights applied, because that's how they're constructed in mapmaking, and because the operation of removing weights can be lossy for poorly conditioned polarization data.  Here we're just dealing with temperature data, so these operations are greatly simplified.

Let's take a look at the temperature map in the frame:

In [ ]:
m = frame["T"]
print(m)
print("Shape:", m.shape)
print("Size:", m.size)
print("Allocated:", m.npix_allocated)
print("Fraction allocated:", m.npix_allocated / m.size)
print("Sparse?", m.sparse)

In [ ]:
m2 = m[100:200, 100:200]

In [ ]:
print(m2.pixel_to_angle(0, 0))
print(m.pixel_to_angle(100, 100))

Notice that this map is sparsely populated -- only 18% of the pixels in the map are "allocated", meaning that they are filled with data that is likely non-zero.  This is a memory-efficient way of storing what is otherwise a pretty large 2D array.  You'll want to be careful with what kinds of operations you do on this array to avoid increasing the memory usage significantly.

One of the first operations we would do on such a frame is to remove the weights from the Stokes maps (T, and Q/U if present).  We do this with a canned function* that takes a frame as an input argument.  We use the `zero_nans` argument to avoid populating all of the currently unallocated pixels with `NaN` values:

In [ ]:
maps.RemoveWeights(frame, zero_nans=True)

Notice that our overall memory usage hasn't increased since we loaded the map into memory.  Let's plot this map up:

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
plt.imshow(m, vmin=-200 * core.G3Units.uK, vmax=200 * core.G3Units.uK);

Notice that our memory usage has gone up significantly.  Why?  Well, this plotting function has implicitly constructed a "dense" numpy array out of the map data.

In [ ]:
print(m.sparse)
print("Fraction allocated:", m.npix_allocated / m.size)

Let's turn this back into a compact map:

In [ ]:
m.compact()
print(m.sparse)
print("Fraction allocated:", m.npix_allocated / m.size)

... and do some additional garbage collection (likely something internal to matplotlib)

In [ ]:
import gc
gc.collect()

We can also add two maps together, provided that they are both built on the same "footprint" or "stub", i.e. they use the same coordinate transformation.  For example, let's add another subfield to our map:

In [ ]:
frame2 = list(cluster.GridFile("/sptgrid/data/onlinemaps/ra0hdec-52.25/294249241_150GHz_tonly.g3.gz"))[-1]
maps.RemoveWeights(frame2, zero_nans=True)
coadd = m + frame2["T"]

Notice that this coadded map is also sparsely sampled, so this map addition operation preserves the sparsity of the input map.

In [ ]:
print(coadd.sparse)
print(coadd.npix_allocated / coadd.size)

In [ ]:
plt.imshow(coadd, vmin=-200 * core.G3Units.uK, vmax=200 * core.G3Units.uK);

In [ ]:
coadd.compact()
gc.collect()

All of the typical arithmetic operations that one can do with map objects have been written in a way that tries to preserve the sparsity of the map as much as possible.  Map objects also have methods for most of the common numpy operations that one might do on an array, which can optionally avoid zeroes (empty pixels) and/or NaN values.  See the documentation for details.  The maps [tests directory](https://github.com/SouthPoleTelescope/spt3g_software/tree/master/maps/tests) also contains several scripts for testing various functionality so you can see how many of the methods and functions can be used there.